# 03 — Golden set propio

El §4 del enunciado pide **20 preguntas nuestras, de las cuales al menos 6
comparativas**, en el mismo esquema que las 20 oficiales. Este notebook las
construye, las valida y las escribe en `golden_set_propio.jsonl`.

## Por qué un golden set y no «probar unas cuantas preguntas»

Sin un conjunto fijo de preguntas con respuesta conocida, «he mejorado el
sistema» es una impresión. Con él, es un número que se puede comparar antes y
después, y una afirmación que otro puede reproducir. Todo lo que viene en los
notebooks 04, 05 y 06 —los experimentos de retrieval, la tabla baseline
contra final, el informe— se apoya en este fichero. Si está mal, todo lo
demás mide mal y además lo hace en silencio.

## Las tres decisiones que gobiernan cómo se ha escrito

**Primera: la verdad de una pregunta extractiva es una frase, no un
`chunk_id`.** El notebook 04 cambia el troceado, y en cuanto se toca la
ventana o el solape todos los identificadores son otros. Una métrica anclada
al identificador daría recall cero justo en el experimento que se hizo para
mejorarlo, y penalizaría al que lo mejora. Por eso `ancla_texto` es un texto
literal del informe, con sus desplazamientos.

**Segunda: ninguna cifra se escribe a mano.** Las `cifra_esperada` se leen de
`corpus/xbrl_facts.parquet` en tiempo de construcción. Copiar a mano veinte
números de doce dígitos garantiza al menos una errata, y una errata en el
golden set no se manifiesta como un error: se manifiesta como un sistema que
parece fallar cuando en realidad acierta.

**Tercera: ningún ancla se escribe a mano tampoco.** Se indica un *marcador*
—un fragmento corto y distintivo de la frase— y el código localiza la frase
completa en el texto reconstruido, calcula sus desplazamientos exactos y
resuelve el `chunk_id` que la contiene. Escribir los desplazamientos a mano
tiene el mismo problema que escribir las cifras, y peor: un desfase de tres
caracteres no se ve mirando el fichero.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
for carpeta in (RAIZ, RAIZ / "modulos"):
    if str(carpeta) not in sys.path:
        sys.path.insert(0, str(carpeta))

import json

import pandas as pd

from agente import config, corpus, esquema

secciones = {
    (s["ticker"], s["fiscal_year"], s["item"]): s for s in corpus.cargar_secciones()
}
chunks = corpus.cargar_chunks()
xbrl = corpus.cargar_xbrl()
print(f"{len(secciones)} secciones · {len(chunks)} fragmentos · {len(xbrl)} hechos XBRL")

48 secciones · 1749 fragmentos · 137 hechos XBRL


## 1. De dónde salen las anclas y las cifras

La idea que gobierna todo el notebook es que **nada que pueda leerse del
corpus se escribe a mano**. Ni una cifra, ni un desplazamiento, ni un
`chunk_id`. Lo único que se teclea es la pregunta, el emisor, el ejercicio y
un *marcador*: las primeras palabras de la frase del informe que contiene la
respuesta.

El motivo es que una errata en un ancla —un espacio de más, unas comillas
tipográficas cambiadas— **no da ningún error**. Hace que la pregunta sea
imposible de acertar y hunde el `recall` medido por un motivo que no tiene
nada que ver con el retriever. Un golden set con tres erratas mide el golden
set, no el sistema.

Las funciones que implementan eso están en `agente/golden.py` y no en este
notebook, porque el notebook 07 construye un segundo conjunto —el hold-out
simulado— con la misma mecánica. La celda siguiente las importa y comprueba
sobre un ejemplo que el mecanismo hace lo que dice.


In [2]:
# Las cuatro funciones que siguen viven en `agente/golden.py` y no aquí. El
# motivo es el notebook 07: construye un segundo conjunto de preguntas —el
# hold-out simulado— con exactamente la misma mecánica, y dos copias de un
# extractor de anclas se separan en cuanto alguien toca una de las dos.
#
# Lo que hace cada una:
#
#   resolver_ancla(ticker, fy, item, marcador, fin_marcador=None)
#       Localiza la frase que contiene el marcador y devuelve su texto, sus
#       desplazamientos dentro de la sección y el chunk_id que la contiene
#       entera. Lanza si el marcador no es unico en la seccion: un ancla
#       ambigua contaria como acierto un fragmento de otro sitio del documento.
#       `fin_marcador` recorta la frase cuando pasa de 40 palabras, que es el
#       tope del validador oficial. El recorte sigue siendo texto literal.
#
#   cifra_de(ticker, fy, concepto)
#       El valor y la unidad, leidos del parquet de hechos XBRL.
#
#   construir(spec)
#       Una entrada completa del golden set a partir de su especificacion.
#
#   escribir(ruta, filas, campos)
#       Volcado a JSONL con las claves en orden canonico.
from agente.golden import (
    MAXIMO_PALABRAS,
    cifra_de,
    construir,
    en_millones,
    escribir,
    resolver_ancla,
)

# Una comprobacion de que el corpus esta montado y de que el mecanismo del
# ancla hace lo que dice, antes de construir veinte preguntas con el.
demostracion = resolver_ancla("AAPL", 2025, "1A", "Beginning in the second quarter of 2025")
print(f"Ancla de ejemplo, resuelta contra el texto reconstruido:\n")
print(f"  texto : {demostracion['ancla_texto']}")
print(f"  offset: [{demostracion['ancla_inicio']}, {demostracion['ancla_fin']})")
print(f"  chunk : {demostracion['chunk_id_esperado']}")
print(f"\nMaximo de palabras por ancla que impone el validador: {MAXIMO_PALABRAS}")


Ancla de ejemplo, resuelta contra el texto reconstruido:

  texto : Beginning in the second quarter of 2025, new tariffs were announced on imports to the U.S.
  offset: [6147, 6237)
  chunk : AAPL-2025-1A-0003

Maximo de palabras por ancla que impone el validador: 40


## 2. La especificación de las 20 preguntas

Se escribe lo mínimo indispensable —qué se pregunta, sobre qué compañía,
ejercicio y concepto, y dónde está la frase que lo respalda— y el código
rellena el resto.

### Cómo se ha repartido la cobertura, y por qué así

El golden set tiene que ejercitar el sistema, no lucirse con preguntas
fáciles. Los criterios de reparto:

- **Las seis compañías y los dos ejercicios.** Si una compañía faltara, el
  golden set no detectaría un fallo específico de ella, y hay fallos
  específicos de compañía en este corpus: el concepto de ingresos de NVIDIA
  no es el de Apple.
- **Las cuatro secciones.** El Item 7A son 37 fragmentos de 1.749 y sin
  filtro no compite con nada; si no hubiera ninguna pregunta sobre él, el
  golden set no mediría el arreglo que más lo beneficia.
- **Las comparativas cruzan los dos ejercicios de verdad**, con la cifra de
  los dos años y con la frase del MD&A que explica la variación. Una
  comparativa que se pueda contestar con una sola consulta no es una
  comparativa.

### Por qué las comparativas son la familia que importa

«¿Cuánto creció el revenue de NVIDIA?» no la contesta una sola recuperación:
hay que descomponer en dos consultas, recuperar dos veces y comparar. Es la
familia donde el agente deja de ser decoración sobre una *pipeline*, y sin
ella el informe no puede demostrar que hiciera falta un agente.

Dos de las seis están elegidas porque su respuesta es **contraintuitiva**, y
eso las hace especialmente buenas para detectar un sistema que razona por
analogía en lugar de consultar: el beneficio neto de Meta **cayó** en FY2025
aunque los ingresos subieron, por el efecto fiscal de la OBBBA; y el flujo de
caja libre de Amazon se desplomó pese a que el flujo operativo creció, por el
capex. Un agente que estime en vez de consultar se equivoca en las dos.

In [3]:
ESPECIFICACION = [
    # ================= EXTRACTIVAS (7) =====================================
    {
        "id": "gp-001",
        "familia": "extractiva",
        "pregunta": "¿Qué dice NVIDIA en sus factores de riesgo de FY2025 sobre los controles de exportación aplicados a sus productos de red?",
        "ticker": "NVDA",
        "fiscal_year": 2025,
        "item": "1A",
        "marcador": "export controls on our networking products",
        "respuesta_esperada": "NVIDIA advierte de que los controles de exportación sobre sus productos de red, como las interconexiones de alta velocidad, buscan limitar la capacidad de terceros de construir grandes clústeres para entrenar modelos frontera, y de que cualquier control nuevo que alcance a más productos suyos le perjudicaría.",
    },
    {
        "id": "gp-002",
        "familia": "extractiva",
        "pregunta": "¿De qué depende la expansión de los centros de datos de Microsoft según los factores de riesgo de su 10-K de FY2025?",
        "ticker": "MSFT",
        "fiscal_year": 2025,
        "item": "1A",
        "marcador": "Our datacenters depend on the availability of permitted and buildable land",
        "respuesta_esperada": "Microsoft declara que sus centros de datos dependen de la disponibilidad de suelo permitido y edificable, de energía previsible y de suministros de red, entre otros factores.",
    },
    {
        "id": "gp-003",
        "familia": "extractiva",
        "pregunta": "¿Qué normativa europea cita Meta en sus factores de riesgo de FY2025 como fuente de incertidumbre regulatoria?",
        "ticker": "META",
        "fiscal_year": 2025,
        "item": "1A",
        "marcador": "Digital Markets Act (DMA), Digital Services Act (DSA)",
        "respuesta_esperada": "Meta cita la Digital Markets Act, la Digital Services Act, la Online Safety Act del Reino Unido, el Reglamento de Inteligencia Artificial de la UE y la DMCC británica.",
    },
    {
        "id": "gp-004",
        "familia": "extractiva",
        "pregunta": "¿Qué cambios dice Apple haber introducido en la App Store y en Safari en la Unión Europea para cumplir con la Digital Markets Act, según su 10-K de FY2025?",
        "ticker": "AAPL",
        "fiscal_year": 2025,
        "item": "1A",
        "marcador": "App Store and Safari® in the EU as it seeks to comply",
        "fin_marcador": "alternative methods of distribution for iOS and iPadOS apps,",
        "respuesta_esperada": "Apple describe nuevos términos comerciales y estructuras de comisiones alternativas para las aplicaciones de iOS e iPadOS, métodos alternativos de distribución y de procesamiento de pagos, y cambios en los navegadores.",
    },
    {
        "id": "gp-005",
        "familia": "extractiva",
        "pregunta": "¿Qué resolvió el tribunal del Distrito de Columbia en el caso antimonopolio contra Google y qué sentencia firme se dictó, según el 10-K de Alphabet de FY2025?",
        "ticker": "GOOGL",
        "fiscal_year": 2025,
        "item": "1A",
        "marcador": "In August 2024, the US District Court for the District of Columbia ruled against Google",
        "fin_marcador": "entered a final judgment requiring remedies,",
        "respuesta_esperada": "El tribunal falló contra Google en agosto de 2024 y en diciembre de 2025 dictó sentencia firme imponiendo remedios que restringen, entre otras cosas, cómo distribuye Google sus servicios.",
    },
    {
        "id": "gp-006",
        "familia": "extractiva",
        "pregunta": "¿Cuál era el saldo de Amazon en fondos denominados en divisa extranjera a cierre de 2025 y qué sensibilidad declara ante una variación adversa del tipo de cambio?",
        "ticker": "AMZN",
        "fiscal_year": 2025,
        "item": "7A",
        "marcador": "Based on the balance of foreign funds as of December 31, 2025",
        "respuesta_esperada": "Amazon declara 29.700 millones de dólares en fondos en divisa extranjera a 31 de diciembre de 2025, y cuantifica el efecto de variaciones adversas del 5 %, 10 % y 20 % del tipo de cambio.",
    },
    {
        "id": "gp-007",
        "familia": "extractiva",
        "pregunta": "¿Cómo define Apple su ejercicio fiscal en las notas a los estados financieros de FY2025?",
        "ticker": "AAPL",
        "fiscal_year": 2025,
        "item": "8",
        "marcador": "52- or 53-week period that ends on the last Saturday of September",
        "respuesta_esperada": "Apple define su ejercicio fiscal como un periodo de 52 o 53 semanas que termina el último sábado de septiembre, con una semana adicional en el primer trimestre cada cinco o seis años.",
    },
    # ================= NUMÉRICAS (7) =======================================
    {
        "id": "gp-008",
        "familia": "numerica",
        "pregunta": "¿Cuál fue el beneficio neto de Apple en el ejercicio fiscal 2024?",
        "ticker": "AAPL",
        "fiscal_year": 2024,
        "concepto": "NetIncomeLoss",
    },
    {
        "id": "gp-009",
        "familia": "numerica",
        "pregunta": "¿Cuánto gastó Microsoft en investigación y desarrollo en el ejercicio fiscal 2024?",
        "ticker": "MSFT",
        "fiscal_year": 2024,
        "concepto": "ResearchAndDevelopmentExpense",
    },
    {
        "id": "gp-010",
        "familia": "numerica",
        "pregunta": "¿Cuál era el activo total de Alphabet al cierre del ejercicio fiscal 2024?",
        "ticker": "GOOGL",
        "fiscal_year": 2024,
        "concepto": "Assets",
    },
    {
        "id": "gp-011",
        "familia": "numerica",
        "pregunta": "¿Cuál fue el beneficio por acción diluido de NVIDIA en el ejercicio fiscal 2025?",
        "ticker": "NVDA",
        "fiscal_year": 2025,
        "concepto": "EarningsPerShareDiluted",
    },
    {
        "id": "gp-012",
        "familia": "numerica",
        "pregunta": "¿A cuánto ascendía el patrimonio neto de Amazon al cierre del ejercicio fiscal 2024?",
        "ticker": "AMZN",
        "fiscal_year": 2024,
        "concepto": "StockholdersEquity",
    },
    {
        "id": "gp-013",
        "familia": "numerica",
        "pregunta": "¿Cuál fue el resultado de explotación de Meta en el ejercicio fiscal 2024?",
        "ticker": "META",
        "fiscal_year": 2024,
        "concepto": "OperatingIncomeLoss",
    },
    {
        "id": "gp-014",
        "familia": "numerica",
        "pregunta": "¿Cuánto invirtió Microsoft en inmovilizado material en el ejercicio fiscal 2025?",
        "ticker": "MSFT",
        "fiscal_year": 2025,
        "concepto": "PaymentsToAcquirePropertyPlantAndEquipment",
    },
    # ================= COMPARATIVAS (6) ====================================
    {
        "id": "gp-015",
        "familia": "comparativa",
        "pregunta": "¿Cuánto crecieron los ingresos de NVIDIA entre los ejercicios fiscales 2024 y 2025, y a qué lo atribuye la dirección?",
        "ticker": "NVDA",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "Revenues",
        "item": "7",
        "marcador": "Revenue from Data Center computing grew 162%",
        "respuesta_esperada": "Los ingresos pasaron de 60.922 a 130.497 millones de dólares, algo más del doble. La dirección lo atribuye sobre todo al negocio de centro de datos, cuyo cómputo creció un 162 % por la demanda de la plataforma Hopper para modelos de lenguaje, motores de recomendación e IA generativa.",
    },
    {
        "id": "gp-016",
        "familia": "comparativa",
        "pregunta": "¿Cómo evolucionó el margen bruto de Microsoft entre los ejercicios fiscales 2024 y 2025, y qué explicación da la dirección?",
        "ticker": "MSFT",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "GrossProfit",
        "item": "7",
        "marcador": "Gross margin increased $22.9 billion or 13%",
        "respuesta_esperada": "El margen bruto creció 22.900 millones de dólares, un 13 %, con crecimiento en todos los segmentos. En porcentaje sobre ingresos bajó ligeramente, por el efecto de escalar la infraestructura de IA.",
    },
    {
        "id": "gp-017",
        "familia": "comparativa",
        "pregunta": "¿Cuánto aumentó la inversión de Alphabet en inmovilizado material entre 2024 y 2025, y cómo lo explica la compañía?",
        "ticker": "GOOGL",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "PaymentsToAcquirePropertyPlantAndEquipment",
        "item": "7",
        "marcador": "Net cash used in investing activities increased from 2024 to 2025",
        "respuesta_esperada": "La inversión en inmovilizado pasó de 52.535 a 91.351 millones de dólares. Alphabet lo atribuye a las inversiones en infraestructura técnica, que son también la causa principal del aumento de la salida de caja por inversión.",
    },
    {
        "id": "gp-018",
        "familia": "comparativa",
        "pregunta": "¿Subió o bajó el beneficio neto de Meta entre 2024 y 2025, y por qué?",
        "ticker": "META",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "NetIncomeLoss",
        "item": "7",
        "marcador": "Effective tax rate was 30% for the year ended December 31, 2025",
        "respuesta_esperada": "Bajó, de 62.360 a 60.458 millones de dólares, pese a que los ingresos crecieron. La causa es fiscal: el tipo efectivo subió al 30 % por la entrada en vigor de la One Big Beautiful Bill Act, frente al 12 % del año anterior.",
    },
    {
        "id": "gp-019",
        "familia": "comparativa",
        "pregunta": "¿Cómo varió el margen bruto de Apple entre los ejercicios 2024 y 2025, y qué razones da la dirección?",
        "ticker": "AAPL",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "GrossProfit",
        "item": "7",
        "marcador": "Products gross margin increased during 2025 compared to 2024",
        "respuesta_esperada": "El margen bruto subió de 180.683 a 195.201 millones de dólares. En productos, la dirección lo atribuye a costes favorables y a un mix distinto de productos, parcialmente compensado por los costes arancelarios.",
    },
    {
        "id": "gp-020",
        "familia": "comparativa",
        "pregunta": "¿Cuánto creció el flujo de caja de las operaciones de Amazon entre 2024 y 2025?",
        "ticker": "AMZN",
        "fiscal_year": 2025,
        "fiscal_year_anterior": 2024,
        "concepto": "NetCashProvidedByUsedInOperatingActivities",
        "item": "7",
        "marcador": "operating activities was $115.9 billion and $139.5 billion in 2024 and 2025",
        "respuesta_esperada": "Pasó de 115.877 a 139.514 millones de dólares, un crecimiento en torno al 20 %. La propia compañía lo resume como 115.900 y 139.500 millones en 2024 y 2025.",
    },
]

print(f"{len(ESPECIFICACION)} preguntas especificadas")
print(pd.Series([e["familia"] for e in ESPECIFICACION]).value_counts().to_string())

20 preguntas especificadas
extractiva     7
numerica       7
comparativa    6


## 3. Construcción: el código rellena lo que falta

A partir de aquí no se escribe ningún número ni ningún desplazamiento a mano.
Las cifras salen de `xbrl_facts.parquet` y los anclas de `secciones.jsonl`.

In [4]:
golden_propio = [construir(s) for s in ESPECIFICACION]
print(f"{len(golden_propio)} preguntas construidas sin erratas de transcripcion: "
      f"ninguna cifra ni ningun offset se ha escrito a mano.")


20 preguntas construidas sin erratas de transcripcion: ninguna cifra ni ningun offset se ha escrito a mano.


## 4. Validación

Se pasa el validador oficial —el de la celda 32 del notebook de la sesión 1,
movido a `agente/esquema.py` para que el notebook que construye el fichero y
el que lo consume apliquen el mismo criterio— y después las comprobaciones
adicionales de cobertura.

In [5]:
problemas = esquema.validar_golden(golden_propio, exigir_20=True)
print("VALIDADOR OFICIAL")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")
assert not problemas, problemas

VALIDADOR OFICIAL
  sin problemas


In [6]:
# --- Cobertura -------------------------------------------------------------
tabla = pd.DataFrame(golden_propio)

print("REPARTO POR FAMILIA")
print(tabla.familia.value_counts().to_string())
print("\nCOMPAÑÍAS × EJERCICIO")
print(pd.crosstab(tabla.ticker, tabla.fiscal_year).to_string())
print("\nSECCIÓN ESPERADA (las numéricas no tienen: se contestan con XBRL)")
print(tabla.item_esperado.value_counts(dropna=False).to_string())
print("\nCONCEPTOS XBRL EJERCITADOS")
print(tabla.concept_xbrl.value_counts(dropna=False).to_string())

assert set(tabla.ticker) == set(config.TICKERS), "Falta alguna compañía."
assert set(tabla.fiscal_year) == set(config.EJERCICIOS), "Falta algún ejercicio."
items_cubiertos = set(tabla.item_esperado.dropna())
assert items_cubiertos == set(config.ITEMS), (
    f"Los items cubiertos son {sorted(items_cubiertos)} y tienen que ser los "
    f"cuatro: {config.ITEMS}. El 7A son 37 fragmentos de 1.749 y si no hay "
    f"ninguna pregunta sobre él, el golden set no mide el arreglo que más lo "
    f"beneficia."
)
assert (tabla.familia == "comparativa").sum() >= 6
print("\nCobertura completa: 6 compañías, 2 ejercicios, 4 secciones.")

REPARTO POR FAMILIA
familia
extractiva     7
numerica       7
comparativa    6

COMPAÑÍAS × EJERCICIO
fiscal_year  2024  2025
ticker                 
AAPL            1     3
AMZN            1     2
GOOGL           1     2
META            1     2
MSFT            1     3
NVDA            0     3

SECCIÓN ESPERADA (las numéricas no tienen: se contestan con XBRL)
item_esperado
NaN    7
7      6
1A     5
7A     1
8      1

CONCEPTOS XBRL EJERCITADOS
concept_xbrl
NaN                                           7
NetIncomeLoss                                 2
PaymentsToAcquirePropertyPlantAndEquipment    2
GrossProfit                                   2
ResearchAndDevelopmentExpense                 1
Assets                                        1
EarningsPerShareDiluted                       1
StockholdersEquity                            1
OperatingIncomeLoss                           1
Revenues                                      1
NetCashProvidedByUsedInOperatingActivities    1

Cobertura 

In [7]:
# --- Los anclas, uno a uno -------------------------------------------------
#
# La misma comprobación que valida la reconstrucción del corpus en el notebook
# 00, aplicada ahora a nuestras frases.
print("ANCLAS")
sin_fragmento = []
for g in golden_propio:
    if not g["ancla_texto"]:
        continue
    seccion = secciones[(g["ticker"], g["fiscal_year"], g["item_esperado"])]
    recorte = seccion["texto"][g["ancla_inicio"]:g["ancla_fin"]]
    assert recorte == g["ancla_texto"], g["id"]
    if g["chunk_id_esperado"] is None:
        sin_fragmento.append(g["id"])
    print(f"  OK {g['id']}  {g['ticker']} FY{g['fiscal_year']} item "
          f"{g['item_esperado']:>2}  [{g['ancla_inicio']:6d}, {g['ancla_fin']:6d})  "
          f"{len(g['ancla_texto'].split()):2d} palabras  "
          f"{g['chunk_id_esperado'] or 'PARTIDA ENTRE FRAGMENTOS'}")

print(f"\n{sum(1 for g in golden_propio if g['ancla_texto'])} anclas verificadas "
      f"carácter a carácter.")
if sin_fragmento:
    print(f"Anclas que ningún fragmento contiene entera: {sin_fragmento}. No es "
          f"un error del golden set: es un límite del troceado, y el notebook "
          f"04 mide exactamente eso.")

ANCLAS
  OK gp-001  NVDA FY2025 item 1A  [ 84661,  84843)  26 palabras  NVDA-2025-1A-0037
  OK gp-002  MSFT FY2025 item 1A  [ 36945,  37132)  24 palabras  MSFT-2025-1A-0018
  OK gp-003  META FY2025 item 1A  [  2873,   3061)  29 palabras  META-2025-1A-0001
  OK gp-004  AAPL FY2025 item 1A  [ 50431,  50668)  40 palabras  AAPL-2025-1A-0023
  OK gp-005  GOOGL FY2025 item 1A  [ 58261,  58416)  25 palabras  GOOGL-2025-1A-0025
  OK gp-006  AMZN FY2025 item 7A  [  4209,   4426)  38 palabras  AMZN-2025-7A-0003
  OK gp-007  AAPL FY2025 item  8  [  9753,   9819)  12 palabras  AAPL-2025-8-0008
  OK gp-015  NVDA FY2025 item  7  [ 23800,  23993)  27 palabras  NVDA-2025-7-0013
  OK gp-016  MSFT FY2025 item  7  [ 13799,  13883)  14 palabras  MSFT-2025-7-0007
  OK gp-017  GOOGL FY2025 item  7  [ 35475,  35726)  38 palabras  GOOGL-2025-7-0018
  OK gp-018  META FY2025 item  7  [  4999,   5063)  12 palabras  META-2025-7-0003
  OK gp-019  AAPL FY2025 item  7  [  7085,   7245)  25 palabras  AAPL-2025-7-0004

In [8]:
# --- Las cifras, contra el corpus -----------------------------------------
print("CIFRAS")
for g in golden_propio:
    if g["cifra_esperada"] is None:
        continue
    hecho = corpus.hecho_xbrl(g["ticker"], g["fiscal_year"], g["concept_xbrl"])
    assert hecho is not None and hecho["value"] == g["cifra_esperada"], g["id"]
    linea = (f"  OK {g['id']}  {g['ticker']} FY{g['fiscal_year']} "
             f"{g['concept_xbrl'][:44]:44s} {g['cifra_esperada']:>18,.2f} {g['unidad']}")
    if g["cifra_anterior_esperada"] is not None:
        variacion = 100 * (g["cifra_esperada"] / g["cifra_anterior_esperada"] - 1)
        linea += f"  (FY{g['fiscal_year_anterior']}: {g['cifra_anterior_esperada']:>16,.0f}, {variacion:+.0f} %)"
    print(linea)

CIFRAS
  OK gp-008  AAPL FY2024 NetIncomeLoss                                 93,736,000,000.00 USD
  OK gp-009  MSFT FY2024 ResearchAndDevelopmentExpense                 29,510,000,000.00 USD
  OK gp-010  GOOGL FY2024 Assets                                       450,256,000,000.00 USD
  OK gp-011  NVDA FY2025 EarningsPerShareDiluted                                    2.94 USD/shares
  OK gp-012  AMZN FY2024 StockholdersEquity                           285,970,000,000.00 USD
  OK gp-013  META FY2024 OperatingIncomeLoss                           69,380,000,000.00 USD
  OK gp-014  MSFT FY2025 PaymentsToAcquirePropertyPlantAndEquipment    64,551,000,000.00 USD
  OK gp-015  NVDA FY2025 Revenues                                     130,497,000,000.00 USD  (FY2024:   60,922,000,000, +114 %)
  OK gp-016  MSFT FY2025 GrossProfit                                  193,893,000,000.00 USD  (FY2024:  171,008,000,000, +13 %)
  OK gp-017  GOOGL FY2025 PaymentsToAcquirePropertyPlantAndEquipment    91,44

Merece la pena detenerse en dos filas de esa tabla, porque son las que hacen
que este golden set distinga un sistema que consulta de uno que estima:

- **Meta, beneficio neto: −3 %.** Los ingresos de Meta crecieron con fuerza
  en 2025 y su beneficio neto bajó. Un modelo que razone «los ingresos suben,
  luego el beneficio sube» se equivoca de signo. La causa está en el tipo
  impositivo efectivo, que pasó del 12 % al 30 % por la OBBBA, y solo se
  encuentra consultando los dos ejercicios.
- **Alphabet, capex: +74 %.** Es un salto que ningún modelo va a acertar
  estimando, y obliga a las dos llamadas a `get_xbrl_fact`.

## 5. Las preguntas de ausencia, en un fichero aparte

El enunciado avisa de que al menos dos de las diez preguntas ciegas del día
24 tendrán como respuesta correcta **que el dato no está en el corpus**. Es
una capacidad que hay que evaluar, porque es el fallo más caro del dominio:
un agente que rellena huecos con lo más parecido produce respuestas que
parecen correctas y no lo son.

**Por qué en un fichero aparte y no entre las 20.** Por una incompatibilidad
concreta con el validador oficial, no por comodidad: el validador exige
`cifra_esperada` en toda pregunta de familia `numerica` o `comparativa`, y
una pregunta cuya respuesta correcta es «ese dato no existe» no tiene cifra
esperada, por definición. Meterlas entre las 20 obligaría a una de dos cosas:
o inventarles una cifra —que es justo lo que se está evaluando que el agente
no haga— o relajar el validador, que dejaría de detectar el error que está
puesto para detectar. La tercera vía, un fichero aparte con su propia
comprobación, no estropea ninguna de las dos.

**Cómo se evalúan.** Con el mismo `evaluar()`, pero el criterio de acierto es
otro: la respuesta tiene que traer `fuente="ninguna"` **y** `cifra=None`.
Callarse no cuenta como decir que no está.

Los cinco primeros casos son huecos reales de la taxonomía us-gaap que el
enunciado declara; los dos últimos son de otro tipo, compañía y ejercicio
fuera del corpus, y están para comprobar que el agente distingue «no lo
reportó» de «no lo tengo».

In [9]:
AUSENCIAS = [
    {
        "id": "ga-001",
        "pregunta": "¿Cuál fue el margen bruto de Amazon en el ejercicio fiscal 2025?",
        "ticker": "AMZN", "fiscal_year": 2025, "concept_xbrl": "GrossProfit",
        "motivo": "Amazon no etiqueta GrossProfit en us-gaap: presenta el coste de ventas dentro de los gastos operativos, sin subtotal de margen bruto.",
    },
    {
        "id": "ga-002",
        "pregunta": "¿Cuánto gastó Amazon en investigación y desarrollo en el ejercicio fiscal 2025?",
        "ticker": "AMZN", "fiscal_year": 2025, "concept_xbrl": "ResearchAndDevelopmentExpense",
        "motivo": "Amazon reporta 'Technology and infrastructure', que no es el concepto ResearchAndDevelopmentExpense.",
    },
    {
        "id": "ga-003",
        "pregunta": "¿Cuál fue el margen bruto de Meta en el ejercicio fiscal 2024?",
        "ticker": "META", "fiscal_year": 2024, "concept_xbrl": "GrossProfit",
        "motivo": "Meta tampoco publica subtotal de margen bruto en us-gaap.",
    },
    {
        "id": "ga-004",
        "pregunta": "¿A cuánto ascendía el pasivo total de Amazon al cierre del ejercicio 2025?",
        "ticker": "AMZN", "fiscal_year": 2025, "concept_xbrl": "Liabilities",
        "motivo": "Amazon no etiqueta el total de pasivo como concepto independiente.",
    },
    {
        "id": "ga-005",
        "pregunta": "¿Cuánto invirtió NVIDIA en inmovilizado material en el ejercicio fiscal 2025?",
        "ticker": "NVDA", "fiscal_year": 2025, "concept_xbrl": "PaymentsToAcquirePropertyPlantAndEquipment",
        "motivo": "NVIDIA no etiqueta ese concepto a nivel anual en us-gaap.",
    },
    {
        "id": "ga-006",
        "pregunta": "¿Cuáles fueron los ingresos de Tesla en el ejercicio fiscal 2025?",
        "ticker": "AMZN", "fiscal_year": 2025, "concept_xbrl": None,
        "motivo": "Tesla no está en el corpus. El ticker del campo es un relleno: la pregunta es sobre una compañía que no existe aquí.",
        "compania_fuera_del_corpus": "TSLA",
    },
    {
        "id": "ga-007",
        "pregunta": "¿Cuáles fueron los ingresos de Apple en el ejercicio fiscal 2023?",
        "ticker": "AAPL", "fiscal_year": 2025, "concept_xbrl": None,
        "motivo": "El corpus solo tiene FY2024 y FY2025. FY2023 no está, aunque aparezca como comparativo dentro de algunas tablas del informe: esa es justamente la trampa.",
        "ejercicio_fuera_del_corpus": 2023,
    },
]

golden_ausencias = []
for a in AUSENCIAS:
    golden_ausencias.append({
        "id": a["id"],
        "pregunta": a["pregunta"],
        "familia": "numerica",
        "ticker": a["ticker"],
        "fiscal_year": a["fiscal_year"],
        "respuesta_esperada": f"El dato no está en el corpus. {a['motivo']}",
        "cifra_esperada": None,
        "unidad": None,
        "concept_xbrl": a["concept_xbrl"],
        "item_esperado": None,
        "ancla_texto": None,
        "ancla_inicio": None,
        "ancla_fin": None,
        "chunk_id_esperado": None,
        "herramienta_esperada": ["get_xbrl_fact"],
        "respuesta_esperada_es_ausencia": True,
        "autor": config.AUTOR_GOLDEN,
    })

# La comprobación que de verdad importa aquí: que el hueco siga siendo un
# hueco. Si alguno de estos conceptos apareciera en el corpus, la pregunta
# dejaría de ser de ausencia y estaríamos evaluando lo contrario de lo que
# creemos.
print("HUECOS COMPROBADOS CONTRA EL CORPUS")
for a in AUSENCIAS:
    if a["concept_xbrl"] is None:
        print(f"  OK {a['id']}  fuera del corpus por construcción "
              f"({a.get('compania_fuera_del_corpus') or 'FY' + str(a.get('ejercicio_fuera_del_corpus'))})")
        continue
    hecho = corpus.hecho_xbrl(a["ticker"], a["fiscal_year"], a["concept_xbrl"])
    assert hecho is None, (
        f"{a['id']}: {a['concept_xbrl']} SÍ existe para {a['ticker']} "
        f"FY{a['fiscal_year']}. La pregunta ya no es de ausencia."
    )
    print(f"  OK {a['id']}  {a['ticker']} FY{a['fiscal_year']} "
          f"{a['concept_xbrl']:44s} ausente")

HUECOS COMPROBADOS CONTRA EL CORPUS
  OK ga-001  AMZN FY2025 GrossProfit                                  ausente
  OK ga-002  AMZN FY2025 ResearchAndDevelopmentExpense                ausente
  OK ga-003  META FY2024 GrossProfit                                  ausente
  OK ga-004  AMZN FY2025 Liabilities                                  ausente
  OK ga-005  NVDA FY2025 PaymentsToAcquirePropertyPlantAndEquipment   ausente
  OK ga-006  fuera del corpus por construcción (TSLA)
  OK ga-007  fuera del corpus por construcción (FY2023)


## 6. Escritura de los ficheros

In [10]:
escribir(config.RUTA_GOLDEN_PROPIO, golden_propio, esquema.CAMPOS_GOLDEN)
escribir(config.RUTA_GOLDEN_AUSENCIAS, golden_ausencias, esquema.CAMPOS_GOLDEN)

# Relectura: un fichero que no se puede volver a leer no esta escrito.
releido = [json.loads(l) for l in open(config.RUTA_GOLDEN_PROPIO, encoding="utf-8") if l.strip()]
assert len(releido) == 20 and not esquema.validar_golden(releido, exigir_20=True)
print("\nRelectura correcta: el fichero escrito pasa el validador.")


golden_set_propio.jsonl: 20 preguntas, 16.5 KB
golden_set_ausencias.jsonl: 7 preguntas, 4.2 KB

Relectura correcta: el fichero escrito pasa el validador.


In [11]:
# Un ejemplo de cada familia, para poder ver el esquema completo.
for familia in ("extractiva", "numerica", "comparativa"):
    ejemplo = next(g for g in golden_propio if g["familia"] == familia)
    print(f"--- {familia} ---")
    print(json.dumps(ejemplo, ensure_ascii=False, indent=2)[:1100])
    print()

--- extractiva ---
{
  "id": "gp-001",
  "pregunta": "¿Qué dice NVIDIA en sus factores de riesgo de FY2025 sobre los controles de exportación aplicados a sus productos de red?",
  "familia": "extractiva",
  "ticker": "NVDA",
  "fiscal_year": 2025,
  "respuesta_esperada": "NVIDIA advierte de que los controles de exportación sobre sus productos de red, como las interconexiones de alta velocidad, buscan limitar la capacidad de terceros de construir grandes clústeres para entrenar modelos frontera, y de que cualquier control nuevo que alcance a más productos suyos le perjudicaría.",
  "cifra_esperada": null,
  "unidad": null,
  "concept_xbrl": null,
  "item_esperado": "1A",
  "ancla_texto": "export controls on our networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier model training.",
  "ancla_inicio": 84661,
  "ancla_fin": 84843,
  "chunk_id_esperado": "NVDA-2025-1A-0037",
  "herramienta_esperada": [

## Qué queda hecho

| Fichero | Contenido |
| --- | --- |
| `golden_set_propio.jsonl` | 20 preguntas: 7 extractivas, 7 numéricas, 6 comparativas |
| `golden_set_ausencias.jsonl` | 7 preguntas cuya respuesta correcta es que el dato no está |

Las 20 pasan el validador oficial. Los 13 anclas caen carácter a carácter en
sus desplazamientos, las 13 cifras salen del parquet y no de un teclado, y
los 5 huecos de las preguntas de ausencia se han comprobado contra el corpus.

Siguiente: `04_Retrieval_experimentos.ipynb`, que usa estos anclas —y los del
golden set oficial— para medir `recall@5`.